# TinyStories GPT on Runpod
Karpathy architecture, trained from scratch on 20,000 filtered TinyStories. Filtering code and manifest: ../data/tinystories/. Original model dimensions preserved. Run top to bottom.

To rebuild data: run `python scripts/prepare_tinystories.py` from the repository root. See ../data/tinystories/manifest.json for selection rules and provenance. input.txt holds all 20,000 stories; train.txt and val.txt preserve story-level separation. The 256-character context does not cover most whole stories.

Imported v1 snapshot. Saved outputs describe the original Runpod run; paths below were adapted for this repository. Rerunning training replaces the local result files. Checkpoints are GitHub release assets and W&B model artifacts.


In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'data').is_dir() and (p/'notebooks').is_dir())
DATA_DIR = ROOT/'data/tinystories'
OUT_DIR = ROOT/'results/tinystories-5000-step'
CHECKPOINT_DIR = ROOT/'checkpoints/tinystories-5000-step'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
import torch
print(torch.__version__, torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

## 1. TinyStories data and tokenizer
20,000 unique filtered stories: 18,000 train / 2,000 validation. Split by whole story before encoding. input.txt contains all selected stories; training uses train.txt and val.txt. Character-level model with 256-character context.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open(DATA_DIR/'input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
from pathlib import Path
train_data = torch.tensor(encode((DATA_DIR/'train.txt').read_text(encoding='utf-8')), dtype=torch.long)
val_data = torch.tensor(encode((DATA_DIR/'val.txt').read_text(encoding='utf-8')), dtype=torch.long)

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out



In [ ]:
print(len(text), 'characters;', vocab_size, 'symbols')
print('Train chars:', len(train_data), 'Validation chars:', len(val_data))
print(text[:300])

## 2. Model
6 transformer layers, 6 attention heads, 384 embedding dimensions.

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx



In [ ]:
model = GPTLanguageModel().to(device)
print(f'{sum(p.numel() for p in model.parameters()):,} trainable parameters')
print(f'Vocabulary: {vocab_size}; layers: {n_layer}; heads: {n_head}; embedding: {n_embd}; context: {block_size}')

## 3. Train
5,000 steps, batch size 64. Leave the pod and kernel running. Reports loss every 500 steps. Saves checkpoint after training. Exact results vary by hardware.

In [ ]:
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(f'{sum(p.numel() for p in m.parameters()):,} trainable parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

history=[]
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        history.append({k:float(v) for k,v in losses.items()} | {'step':iter})
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


from pathlib import Path
import json
OUT_DIR.mkdir(exist_ok=True)
torch.save({'model':model.state_dict(),'chars':chars,'step':max_iters,'config':dict(n_embd=n_embd,n_head=n_head,n_layer=n_layer,block_size=block_size,dropout=dropout),'optimizer':optimizer.state_dict()},CHECKPOINT_DIR/'checkpoint.pt')
(OUT_DIR/'loss.json').write_text(json.dumps(history))
print('Training complete; checkpoint saved.')


## 4. Learning curves

In [ ]:
from IPython.display import SVG, display
xs=[r['step'] for r in history]
lines=[]
for key,color in [('train','steelblue'),('val','darkorange')]:
    points=' '.join(f'{40+520*x/max(xs):.1f},{250-200*r[key]/max(v[k] for v in history for k in ["train","val"]):.1f}' for x,r in zip(xs,history))
    lines.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2"/>')
display(SVG('<svg xmlns="http://www.w3.org/2000/svg" width="620" height="290"><rect width="620" height="290" fill="white"/><text x="40" y="20">Cross entropy: train (blue), validation (orange)</text>'+''.join(lines)+'<text x="220" y="280">Training step: 0 to 5000</text></svg>'))

## 5. Generate
Change the prompt and rerun. This is a text completion model, not a chat assistant.

In [ ]:
prompt='Once upon a time'
model.eval()
with torch.no_grad():
    context=torch.tensor([encode(prompt)],dtype=torch.long,device=device)
    sample=decode(model.generate(context,max_new_tokens=1000)[0].tolist())
print(sample)
(OUT_DIR/'sample.txt').write_text(sample)

## Reload after a kernel restart
Run data and model definitions first, then use the following optional cell instead of training. Workspace retention depends on Runpod volume settings.

In [ ]:
# ckpt=torch.load(CHECKPOINT_DIR/'checkpoint.pt',map_location=device,weights_only=False)
# model=GPTLanguageModel().to(device)
# model.load_state_dict(ckpt['model'])
# model.eval()